# Esplorazione endpoint Garmin

Obiettivo: capire **quali dati restituisce davvero il tuo account** prima di scrivere il nodo di ingestion.

**Privacy:** la funzione `schema()` stampa solo *nomi dei campi e tipi*, mai i valori → quell'output puoi incollarmelo in sicurezza.
La sezione 5 invece mostra i valori: resta sul tuo PC.

Prerequisiti: `GARMIN_EMAIL` e `GARMIN_PASSWORD` compilati nel file `.env`.

## 1. Setup

In [ ]:
import json
import os
from datetime import date, timedelta
from pathlib import Path

from dotenv import find_dotenv, load_dotenv
from garminconnect import Garmin

ENV_PATH = find_dotenv(usecwd=True)
load_dotenv(ENV_PATH, override=True)  # override: rilegge .env anche se già caricato
ROOT = Path(ENV_PATH).parent          # radice del progetto health-agent
DATA_DIR = ROOT / "data"               # già in .gitignore
DATA_DIR.mkdir(exist_ok=True)

DAY = str(date.today() - timedelta(days=1))   # cambia qui per un altro giorno (YYYY-MM-DD)
print("Progetto:", ROOT)
print("Giorno analizzato:", DAY)
print("Credenziali presenti:", bool(os.getenv("GARMIN_EMAIL")) and bool(os.getenv("GARMIN_PASSWORD")))

## 2. Login

Al primo login i token vengono salvati in `~/.garminconnect` (fuori dal progetto): dai run successivi non servirà ripetere password né MFA.
Se hai la verifica in due passaggi attiva, comparirà un campo per il codice.

In [ ]:
TOKENSTORE = "~/.garminconnect"

api = Garmin(
    os.getenv("GARMIN_EMAIL"),
    os.getenv("GARMIN_PASSWORD"),
    prompt_mfa=lambda: input("Codice MFA: "),
)
api.login(tokenstore=TOKENSTORE)
print("Login OK")

## 3. Helper: struttura senza valori

In [ ]:
def schema(obj, indent=0, max_depth=4):
    """Stampa la struttura (chiavi + tipi) senza mai mostrare i valori."""
    pad = "  " * indent
    if isinstance(obj, dict):
        for k, v in obj.items():
            if isinstance(v, dict) and v and indent < max_depth:
                print(f"{pad}{k}: dict")
                schema(v, indent + 1, max_depth)
            elif isinstance(v, list) and v:
                first = v[0]
                print(f"{pad}{k}: list[{len(v)}] of {type(first).__name__}")
                if isinstance(first, dict) and indent < max_depth:
                    schema(first, indent + 1, max_depth)
            else:
                print(f"{pad}{k}: {type(v).__name__}")
    elif isinstance(obj, list):
        print(f"{pad}list[{len(obj)}]" + (f" of {type(obj[0]).__name__}" if obj else " (vuota)"))
        if obj and isinstance(obj[0], dict):
            schema(obj[0], indent + 1, max_depth)
    else:
        print(f"{pad}{type(obj).__name__}")

## 4. Chiamata agli endpoint candidati

In [ ]:
ENDPOINTS = {
    "sleep":              lambda d: api.get_sleep_data(d),
    "hrv":                lambda d: api.get_hrv_data(d),
    "stats":              lambda d: api.get_stats(d),
    "body_battery":       lambda d: api.get_body_battery(d, d),
    "stress":             lambda d: api.get_stress_data(d),
    "rhr":                lambda d: api.get_rhr_day(d),
    "training_readiness": lambda d: api.get_training_readiness(d),
    "respiration":        lambda d: api.get_respiration_data(d),
    "spo2":               lambda d: api.get_spo2_data(d),
}

raw, status = {}, {}
for name, call in ENDPOINTS.items():
    try:
        raw[name] = call(DAY)
        status[name] = "vuoto" if not raw[name] else "ok"
    except Exception as e:
        raw[name] = None
        status[name] = f"errore: {type(e).__name__}"

for name, s in status.items():
    print(f"{name:<20} {s}")

In [ ]:
# Struttura di ogni endpoint (sicuro da condividere)
for name, data in raw.items():
    if data:
        print(f"\n{'=' * 10} {name} {'=' * 10}")
        schema(data, max_depth=2)

## 5. Campi candidati per `daily_record` (valori: restano sul tuo PC)

Percorsi ipotizzati in base alle versioni note della libreria: qui verifichiamo quali esistono davvero sul tuo account.
Nell'output che mi mandi puoi limitarti alla colonna *trovato sì/no*.

In [ ]:
def dig(obj, *path):
    """Naviga dict/list annidati; ritorna None se il percorso non esiste."""
    for p in path:
        try:
            obj = obj[p]
        except (KeyError, IndexError, TypeError):
            return None
    return obj

CANDIDATES = {
    "sleep_seconds":     ("sleep", "dailySleepDTO", "sleepTimeSeconds"),
    "deep_sleep_s":      ("sleep", "dailySleepDTO", "deepSleepSeconds"),
    "rem_sleep_s":       ("sleep", "dailySleepDTO", "remSleepSeconds"),
    "sleep_score":       ("sleep", "dailySleepDTO", "sleepScores", "overall", "value"),
    "hrv_last_night":    ("hrv", "hrvSummary", "lastNightAvg"),
    "hrv_weekly":        ("hrv", "hrvSummary", "weeklyAvg"),
    "hrv_status":        ("hrv", "hrvSummary", "status"),
    "resting_hr":        ("stats", "restingHeartRate"),
    "stress_avg":        ("stats", "averageStressLevel"),
    "steps":             ("stats", "totalSteps"),
    "bb_max":            ("stats", "bodyBatteryHighestValue"),
    "bb_min":            ("stats", "bodyBatteryLowestValue"),
    "bb_charged":        ("body_battery", 0, "charged"),
    "bb_drained":        ("body_battery", 0, "drained"),
    "readiness_score":   ("training_readiness", 0, "score"),
    "active_kcal":       ("stats", "activeKilocalories"),
    "intensity_min_mod": ("stats", "moderateIntensityMinutes"),
    "intensity_min_vig": ("stats", "vigorousIntensityMinutes"),
}

for field, (src, *path) in CANDIDATES.items():
    val = dig(raw.get(src), *path)
    found = "sì" if val is not None else "NO"
    print(f"{field:<20} {found:<4} {val if val is not None else ''}")

## 6. Salva un campione in locale

Così puoi sviluppare offline: `python main.py --dry-run --sample data/garmin_sample_<data>.json --date <data>` usa questo file invece di chiamare Garmin.

In [ ]:
out = DATA_DIR / f"garmin_sample_{DAY}.json"
out.write_text(json.dumps(raw, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print("Salvato in", out)

## Cosa mandarmi

1. L'output della cella **status** (sezione 4)
2. La colonna *trovato sì/no* della sezione 5
3. Opzionale: l'output di `schema()` degli endpoint dove un campo è risultato NO, così trovo il percorso giusto